# 動作 × 心率負荷整合（PAMAP2）

**核心問題：同一群人做不同動作時，心臟到底付出多少？**

PAMAP2 提供 9 位受試者、18 種活動的**同步**心率與 IMU 量測——
不必跨資料集拼貼，直接算出每個動作的平均 %HRmax（負荷梯）。


In [ ]:
import sys; sys.path.insert(0, '..')
from src.isa.config import RAW_DIR, DEFAULT_MAX_HR
from src.isa.data.loader_pamap2 import load_all_protocol, PAMAP2_ACTIVITIES

hits = list(RAW_DIR.glob('**/subject101.dat'))
df = load_all_protocol(hits[0].parent)
df['pct_hrmax'] = df['hr'] / DEFAULT_MAX_HR * 100
df.head()

## 逐活動負荷梯

平均 %HRmax 排序——哪些日常動作其實是「隱形的中強度運動」？
與 `scripts/run_integration.py` 同一份分析（可重跑驗證）。


In [ ]:
agg = (df[df['activity'].isin(PAMAP2_ACTIVITIES)]
        .groupby('activity')
        .agg(pct=('pct_hrmax', 'mean'), n=('pct_hrmax', 'size'))
        .sort_values('pct'))
agg.index = [PAMAP2_ACTIVITIES[a] for a in agg.index]
agg.plot.barh(y='pct', figsize=(8, 6), legend=False,
              color='#2a78d6', title='Activity load ladder (mean %HRmax)')

## 敘事觀點（生醫×舞蹈）

- 靜態動作（躺/坐/站）佔 %HRmax 低檔——動作識別的正確率瓶頸
  （SITTING↔STANDING）恰好落在生理訊號最無區別力的區段
- 家務（吸地/熨衣）落在輕-中強度：日常累積負荷被低估的證據
- 跳繩/跑步逼近無氧閾值——疲勞與受傷風險分層的實證基礎

限制：HRmax 以 190bpm 估計（PAMAP2 受試者 24-32 歲）。
